## Környezet ellenőrzése

In [ ]:
!conda info --envs

## Notebook tartalomjegyzéke

Felhasznált adathalmaz: **Online Retail II UCI**
https://www.kaggle.com/datasets/mashlyn/online-retail-ii-uci/data  
0\. Adatbetöltés  
1\. Adattisztítás  
2\. RFM feature engineering  
3\. Outlier-kezelés  
4\. EDA - feltáró adatelemzés  
5\. K-means klaszterezés  
6\. XGBoost célváltozó tervezés és modellezés  
7\. SHAP (magyarázhatóság)  
8\. Modell kiértékelés, üzleti elemzés  
9\. Export → processed/, kimentés streamlit dashboardhoz  

## Műszaki és verziókezelési megjegyzések

**Adatformátum (Parquet):** A projekt során a nyers CSV adatokat az első lépésben Apache Parquet formátumba transzformálom és a data/processed/ mappában tárolom az alábbi, data engineeringben ismert előnyök miatt:


Hatékonyság: Az oszlop-alapú tárolás jelentősen gyorsabb beolvasást és kisebb memóriahasználatot tesz lehetővé a feature engineering során.


Típusbiztonság: A Parquet megőrzi a sémát (pl. InvoiceDate datetime vagy Customer ID integer típusát), így elkerülhetők a CSV-re jellemző ismételt típuskonverziós hibák.

**Verziókezelés (nbstripout):** A notebook tisztán tartása érdekében ``nbstripout`` használatával dolgozom. Ez a git-filter biztosítja, hogy a távoli repóba (GitHub) kizárólag a forráskód kerüljön be, a futtatási metaadatok és nagyméretű vizuális kimenetek nélkül. Ez elősegíti a tiszta code review folyamatot és megakadályozza a bináris szemét felhalmozódását a verziótörténetben.

---
## 0. Adatbetöltés és Parquet-konverzió

A nyers adathalmaz betöltése közvetlenül az **UCI Machine Learning Repository** hivatalos API-ján keresztül történik az `ucimlrepo` csomag segítségével. Ez feleslegessé teszi a nagyméretű CSV fájlok manuális letöltését és verziókövetését.

A `0.2` cella idempotens: ha a tisztított Parquet fájl már létezik, automatikusan kihagyja az ismételt letöltést és konverziót.

In [ ]:
# ============================================================
# 0.1 – Könyvtárak, konstansok és adat-ellenőrzés
# ============================================================
from pathlib import Path
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# Útvonalak beállítása
PROJECT_ROOT = Path(".").resolve()
RAW_DIR       = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Mappastruktúra létrehozása
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Fájlnevek
RAW_FILE = RAW_DIR / "online_retail_II.csv"
PARQUET_OUT = PROCESSED_DIR / "online_retail_raw.parquet"

print(f"PROJECT_ROOT:  {PROJECT_ROOT}")
print(f"RAW_FILE:      {RAW_FILE}")
print(f"PARQUET_OUT:   {PARQUET_OUT}\n")

# --- Klónozás utáni állapot ellenőrzése ---
if not PARQUET_OUT.exists() and not RAW_FILE.exists():
    error_msg = (
        "\n" + "="*80 + "\n"
        "HIÁNYZÓ ADATHALMAZ!\n"
        "A hivatalos UCI API nem támogatja ezt a specifikus adathalmazt, így manuális letöltés szükséges.\n\n"
        "Kérlek, kövesd az alábbi lépéseket a folytatáshoz:\n"
        "1. Töltsd le a zip fájlt a Kaggle-ről:\n"
        "   https://www.kaggle.com/datasets/mashlyn/online-retail-ii-uci/data\n"
        "2. Csomagold ki, és az 'online_retail_II.csv' fájlt helyezd el ide:\n"
        f"   {RAW_FILE}\n"
        "3. Futtasd újra ezt a cellát!\n"
        + "="*80 + "\n"
    )
    print(error_msg)
    raise FileNotFoundError("A nyers adathalmaz nem található. Kövesd a fenti utasításokat!")
else:
    print("Fájlrendszer ellenőrizve: a szükséges adatfájlok rendelkezésre állnak!")

In [ ]:
# ============================================================
# 0.2 – Nyers CSV betöltése és Parquet-be konvertálása
# ============================================================

if PARQUET_OUT.exists():
    print(f"Parquet már létezik, kihagyjuk a konverziót: {PARQUET_OUT}")
else:
    dtype_map = {
        "Invoice":     "string",
        "StockCode":   "string",
        "Description": "string",
        "Quantity":    "float64",
        "Price":       "float64",
        "Customer ID": "float64",
        "Country":     "string",
    }
    parse_dates = ["InvoiceDate"]

    print(f"CSV fájl betöltése innen: {RAW_FILE} ... (ez eltarthat egy percig)")
    df_raw = pd.read_csv(
        RAW_FILE,
        dtype=dtype_map,
        parse_dates=parse_dates,
        encoding="utf-8",
    )
    
    print(f"Összesített sorok (nyers): {len(df_raw):,}")

    # Customer ID: float -> nullable Int64 (megőrzi a NaN-okat is)
    df_raw["Customer ID"] = df_raw["Customer ID"].astype("Int64")

    # Parquet mentés
    df_raw.to_parquet(PARQUET_OUT, compression="snappy", index=False)

    size_mb = PARQUET_OUT.stat().st_size / 1_048_576
    print(f"\nParquet mentve: {PARQUET_OUT}")
    print(f"Fájlméret:      {size_mb:.1f} MB")
    print(f"Sorok:          {len(df_raw):,} | Oszlopok: {df_raw.shape[1]}")
    print(f"\nSéma:\n{df_raw.dtypes}")

---
## 1. Adattisztítás

A Customer Lifetime Value (CLV) modellezés és az RFM szegmentáció alapfeltétele, hogy az adatokat vásárlói szinten tudjuk aggregálni. Ennek megfelelően az alábbi tisztítási lépéseket végezzük el:
- **Anonim tranzakciók eltávolítása:** A hiányzó `Customer ID`-val rendelkező sorok kötelezően eldobandók.
- **Érvénytelen értékek szűrése:** A 0 vagy negatív `Price` értékkel rendelkező sorok eltávolítása.
- **Duplikátumok szűrése:** Az azonos tartalmú, ismétlődő sorok törlése.

*Megjegyzés: A sztornó/visszaküldött tételeket ('C' prefix az Invoice oszlopban) ebben a fázisban szándékosan megtartjuk, mivel ezek a későbbiekben negatív feature-ként szolgálnak a vásárlói érték számításánál.*